In [18]:
import sys
from pathlib import Path

# Add the project root to Python path
project_root = Path().absolute().parent.parent.parent.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
from src.app.vectorstores.search_factory import SearchServiceFactory
from src.app.core.config import settings

Project root: /Users/ahmedashraf/Documents/GitHub/PhysioAssist-AI


In [19]:
search_service = SearchServiceFactory.create_elasticsearch()
info = search_service.client.info()

print("Connection successful!")
print(f"Cluster name: {info['cluster_name']}")
print(f"ElasticSearch version: {info['version']['number']}")
print("✓ ElasticSearch Service created successfully")

Connection successful!
Cluster name: docker-cluster
ElasticSearch version: 9.2.3
✓ ElasticSearch Service created successfully


In [20]:
test_search_query = "Leg injury"

# Full-text search (BM25)

In [25]:

print("\nTesting Full-Text Search (BM25)...")
print("\nSearch Query:", test_search_query)

results = search_service.full_text_search(
    index_name=settings.INDEX_NAME,
    query=test_search_query,
    fields=["title", "content", "tags"],
    size=3
)

for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    print(f"     Score: {result['score']:.4f}")
    print(f"     Category: {result['source']['tags']}")




Testing Full-Text Search (BM25)...

Search Query: Leg injury

  1. Whiplash Injury
     Score: 2.1700
     Category: ['neck', 'trauma', 'whiplash']

  2. PCL Tear
     Score: 1.4735
     Category: ['knee', 'pcl', 'ligament']

  3. ACL Tear
     Score: 1.4535
     Category: ['knee', 'acl', 'ligament']


# Vector search

In [26]:
print("\nTesting Vector Search (Semantic)...")
print("\nSearch Query:", test_search_query)

query_vector = search_service.embed(test_search_query)

results = search_service.vector_search(
    index_name=settings.INDEX_NAME,
    vector=query_vector,
    vector_field="embedding",
    size=3
)
print(f"Found {len(results)} results:")
for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    print(f"     Score: {result['score']:.4f}")
    print(f"     Tags: {', '.join(result['source']['tags'])}")




Testing Vector Search (Semantic)...

Search Query: Leg injury
Found 3 results:

  1. PCL Tear
     Score: 0.7601
     Tags: knee, pcl, ligament

  2. ACL Tear
     Score: 0.7385
     Tags: knee, acl, ligament

  3. Whiplash Injury
     Score: 0.7324
     Tags: neck, trauma, whiplash


# Hybrid search (BM25 + Vector)

In [27]:
print("\nTesting Full-Text Search (BM25) + Vector Search (Semantic)...")
print("\nSearch Query:", test_search_query)

query_vector = search_service.embed(test_search_query)

results = search_service.hybrid_search(
    index_name=settings.INDEX_NAME,
    query=test_search_query,
    vector=query_vector,
    text_fields=["title", "content", "tags"],
    vector_field="embedding",
    text_weight=0,
    vector_weight=1,
    size=5
)
print(f"Found {len(results)} results:")
for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    # print(f"     Score: {result['score']:.4f}")
    # print(f"     Tags: {', '.join(result['source']['tags'])}")
    # print(f"     Content : {result['source']['content']}...")


Testing Full-Text Search (BM25) + Vector Search (Semantic)...

Search Query: Leg injury
Found 3 results:

  1. PCL Tear

  2. ACL Tear

  3. Whiplash Injury
